In [2]:
# Start your code here!
import os
import pandas as pd
from openai import OpenAI

# Instantiate an API client
client = OpenAI()

# Read in the two CSV files
nasdaq100_ca = pd.read_csv('nasdaq100_CA.csv')
nasdaq100_price_change = pd.read_csv('nasdaq100_price_change.csv')

# Merge the ytd column into nasdaq100_ca
nasdaq100_ca = nasdaq100_ca.merge(nasdaq100_price_change[['symbol', 'ytd']], on='symbol')

# Preview the result
print(nasdaq100_ca.head())

  symbol               name        headQuarter  ...      cik     founded       ytd
0   AAPL         Apple Inc.      Cupertino, CA  ...   320193  1976-04-01  42.99992
1   ABNB             Airbnb  San Francisco, CA  ...  1559720  2008-08-01  68.66902
2   ADBE         Adobe Inc.       San Jose, CA  ...   796343  1982-12-01  57.22723
3   ADSK           Autodesk     San Rafael, CA  ...   769397  1982-01-30  10.02701
4   AMAT  Applied Materials    Santa Clara, CA  ...     6951  1967-11-10  55.46366

[5 rows x 7 columns]


In [3]:
# Create a formatted prompt
prompt = """Classify company {company} into one of the following sectors. Answer only with the sector name: Technology, Consumer Cyclical, Industrials, Utilities, Healthcare, Communication, Energy, Consumer Defensive, Real Estate, or Financial.
"""

# Loop through each company and classify it using OpenAI
for company in nasdaq100_ca["symbol"]:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt.format(company=company)}],
        temperature=0.5
    )
    
    # Extract the sector from the response
    sector = response.choices[0].message.content
    
    # Assign the sector to the correct row in the DataFrame
    nasdaq100_ca.loc[nasdaq100_ca["symbol"] == company, "sector"] = sector

# Check the count of each sector
print(nasdaq100_ca["sector"].value_counts())

Technology            23
Consumer Cyclical      7
Healthcare             6
Energy                 1
Consumer Defensive     1
Name: sector, dtype: int64


In [ ]:
# Create a formatted prompt with the nasdaq100_ca DataFrame as a variable
prompt = f"""Provide summary information about Nasdaq-100 stock performance year to date (YTD) of companies headquartered in CA, recommending the two best sectors and two or more companies per sector.
Company data: {nasdaq100_ca}
"""

# Create an OpenAI request
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}],
    temperature=0.0
)

# Store and print the recommendations
stock_recommendations = response.choices[0].message.content
print(stock_recommendations)

### Summary of Nasdaq-100 Stock Performance YTD for California-Based Companies

As of the current year, California-based companies in the Nasdaq-100 have shown varied performance across different sectors. The two best-performing sectors based on year-to-date (YTD) performance are **Technology** and **Consumer Cyclical**.

#### 1. Technology Sector
The Technology sector has been the standout performer, with several companies achieving significant YTD gains. Here are two of the top companies in this sector:

- **Nvidia (NVDA)**
  - **YTD Performance:** 217.27%
  - Nvidia has been a leader in graphics processing units (GPUs) and artificial intelligence, driving substantial growth.

- **Meta Platforms (META)**
  - **YTD Performance:** 153.78%
  - Meta has seen a strong rebound as it continues to innovate in social media and virtual reality.

Other notable mentions in the Technology sector include:
- **Apple Inc. (AAPL)** - YTD: 43.00%
- **Adobe Inc. (ADBE)** - YTD: 57.23%
- **Advanced Micr